# Exploring numerical stability and conservation laws

**Guided companion notebook to the full tested Python package.** The repository CLI creates the reproducible reports and all canonical output files. This notebook is a sandbox for answering *why* the methods behave differently.

Install from the **repository root** (`python -m pip install -e '.[dev]'`) before launching Jupyter. Units are dimensionless, and `ℏ=1` in quantum models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numerical_dynamics.integrators import euler, rk4
from numerical_dynamics.systems import HarmonicOscillator, DampedOscillator, TwoStateSystem
from numerical_dynamics.analytical import harmonic_exact, quantum_exact
from numerical_dynamics.convergence import oscillator_solver, study_convergence, fit_orders
from numerical_dynamics.extensions import AvoidedCrossing
plt.rcParams.update({"figure.figsize": (8,4), "axes.grid": True, "grid.alpha": .2})

## 1. What does one Euler step do?

For $y'=f(t,y)$, Euler updates $y_{n+1}=y_n+hf(t_n,y_n)$. Verify that the first step for the harmonic oscillator from $(x,v)=(1,0)$ is $(1,-h)$.

**Question:** If the spring force always points toward the center, why might these Euler updates nevertheless gain energy?

In [ ]:
osc = HarmonicOscillator()
h = 0.2
a = euler(osc.rhs, (0, h), [1,0], h)
b = rk4(osc.rhs, (0, h), [1,0], h)
print("Euler one step:", a.y[-1], "energy:", osc.energy(a.y[-1]))
print("RK4 one step:  ", b.y[-1], "energy:", osc.energy(b.y[-1]))
print("Exact state:   ", harmonic_exact(osc,[h])[-1])

## 2. Same physics, different geometry

Compare long-time orbits and relative energy changes. This is one reason a trajectory may look reasonable even when a numerical property is wrong. For canonical geometric methods, the output here has been converted from momentum to velocity by `oscillator_solver`.

In [ ]:
methods = ["euler", "rk4", "symplectic_euler", "velocity_verlet"]
fig, ax = plt.subplots(1,2,figsize=(12,4))
for method in methods:
    r = oscillator_solver(osc,method,80,0.15)
    ax[0].plot(r.y[:,0],r.y[:,1],label=method, linewidth=1)
    ax[1].semilogy(r.t[1:],np.maximum(abs(osc.energy(r.y)[1:]/0.5-1),1e-15),label=method)
ax[0].set(xlabel="x",ylabel="v", title="Phase space")
ax[1].set(xlabel="t",ylabel="Absolute relative energy drift",title="Long-time invariant check")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Why does refining the time step matter?

The CLI studies six time steps; this notebook uses four to stay quick. The `p` values come from regression of **maximum state error** versus `dt` on log-log axes. For very small `dt`, roundoff can eventually dominate.

**Questions:** Why is RK4 expected to have slope near 4? Why might an actual slope be 3.99 rather than exactly 4?

In [ ]:
rows = study_convergence([0.2,0.1,0.05,0.025], repeats=1, t_end=8)
for fit in fit_orders(rows):
    print(f"{fit['experiment']:8} {fit['method']:17} p={fit['slope']:.3f}   R²={fit['r_squared']:.5f}")
fig, ax=plt.subplots()
for method in methods:
    data=sorted((r for r in rows if r['experiment']=='harmonic' and r['method']==method), key=lambda r:r['dt'])
    ax.loglog([r['dt'] for r in data],[r['max_error'] for r in data], 'o-',label=method)
ax.set(xlabel='Time step dt',ylabel='Maximum state error',title='Empirical convergence, harmonic oscillator')
ax.legend(); plt.show()

## 4. Quantum dynamics: phase and norm are different questions

A constant Hermitian 2×2 Hamiltonian has an exact unitary propagator. Euler and RK4 generally violate unitarity. Try changing `dt` from `0.1` to `0.025` and compare both state error and norm drift.

In [ ]:
q = TwoStateSystem(delta=1,coupling=.7)
psi0 = np.array([1.+0j,0j])
fig, axes=plt.subplots(1,2,figsize=(12,4))
for name, solver in [('Euler',euler),('RK4',rk4)]:
    s=solver(q.rhs,(0,20),psi0,0.1)
    exact=quantum_exact(q,s.t)
    axes[0].plot(s.t, abs(s.y[:,1])**2, label=name)
    axes[1].semilogy(s.t, np.maximum(abs(q.probability(s.y)-1),1e-16),label=name)
axes[0].plot(s.t, abs(exact[:,1])**2,'k--',label='Exact P₂')
axes[0].set(xlabel='t',ylabel='State 2 population',title='Coherent population transfer')
axes[1].set(xlabel='t',ylabel='|P₁+P₂−1|', title='Loss of unitarity')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

## 5. Optional avoided crossing: don't confuse a reference with an exact solution

With $H(t)=\begin{pmatrix}at&V\\V&-at\end{pmatrix}$, the Hamiltonian is time dependent. We can still compare our original RK4 integration against a stringent **numerical** SciPy reference, but it is *not* an analytic solution and **not** surface hopping.

In [ ]:
crossing=AvoidedCrossing(rate=.5,coupling=.7)
s=rk4(crossing.rhs,(-6,6),psi0,0.1)
reference=crossing.reference(s.t)
fig,ax=plt.subplots()
ax.plot(s.t,abs(s.y[:,1])**2,label='RK4 state 2')
ax.plot(s.t,abs(reference[:,1])**2,'k--',label='DOP853 reference')
ax.set(xlabel='t',ylabel='Population',title='Population transfer near avoided crossing')
ax.legend(); plt.show()
print('Maximum full-state RK4 error:',np.max(np.linalg.norm(s.y-reference,axis=1)))

## Research notebook questions

1. For the oscillator eigenvalues $\lambda=\pm i\omega$, prove that $|1+h\lambda|>1$ for all $h>0$. How does that relate to the first phase-space plot?
2. A large RK4 step can be within its linear stability region but have terrible phase accuracy. Why does boundedness **not** guarantee convergence?
3. Why is energy conservation *not* a valid diagnostic for the damped oscillator, whereas probability conservation *is* for a Hermitian two-state system?
4. What numerical-convergence and normalization checks would you want to run before interpreting population-transfer results in a more complicated nonadiabatic molecular-dynamics code?

**Next:** run `python -m numerical_dynamics.cli --experiment all` and compare your observations to the CSV tables and the README's interpretation.